In [27]:
# -*- coding: utf-8 -*-
"""Laboratory Work No. 2: Permutation Ciphers"""

'Laboratory Work No. 2: Permutation Ciphers'

In [28]:
# Russian alphabet
RUS_ALPHABET = 'АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ'

In [29]:
def route_encrypt(text, key, m, n):
    # Remove spaces
    text = text.replace(' ', '').upper()

    # Pad with A if needed
    if len(text) < m * n:
        text = text + 'А' * (m * n - len(text))

    # Create table
    table = []
    for i in range(m):
        row = []
        for j in range(n):
            row.append(text[i*n + j])
        table.append(row)

    # Get column order from key
    key = key.upper()
    key_order = []
    for i in range(len(key)):
        pos = RUS_ALPHABET.find(key[i])
        key_order.append((pos, i))
    key_order.sort()

    # Read by columns
    result = ""
    for pos, col in key_order:
        for row in range(m):
            result += table[row][col]

    return result

In [30]:
def route_decrypt(cipher, key, m, n):
    cipher = cipher.upper()

    # Get column order from key
    key = key.upper()
    key_order = []
    for i in range(len(key)):
        pos = RUS_ALPHABET.find(key[i])
        key_order.append((pos, i))
    key_order.sort()

    # Create empty table
    table = [['' for j in range(n)] for i in range(m)]

    # Fill by columns
    idx = 0
    for pos, col in key_order:
        for row in range(m):
            table[row][col] = cipher[idx]
            idx += 1

    # Read by rows
    result = ""
    for row in range(m):
        for col in range(n):
            result += table[row][col]

    # Remove padding
    result = result.rstrip('А')

    return result

In [31]:
def rotate90(matrix):
    # Rotate matrix 90 degrees clockwise
    n = len(matrix)
    result = [[0 for i in range(n)] for j in range(n)]
    for i in range(n):
        for j in range(n):
            result[j][n-1-i] = matrix[i][j]
    return result

In [32]:
def generate_grid(k):
    size = 2 * k

    # Create initial square
    square = []
    num = 1
    for i in range(k):
        row = []
        for j in range(k):
            row.append(num)
            num += 1
        square.append(row)

    # Create big grid
    grid = [[0 for i in range(size)] for j in range(size)]

    # Place rotated squares in four quadrants
    for i in range(k):
        for j in range(k):
            grid[i][j] = square[i][j]

    sq1 = rotate90(square)
    for i in range(k):
        for j in range(k):
            grid[i][j+k] = sq1[i][j]

    sq2 = rotate90(sq1)
    for i in range(k):
        for j in range(k):
            grid[i+k][j] = sq2[i][j]

    sq3 = rotate90(sq2)
    for i in range(k):
        for j in range(k):
            grid[i+k][j+k] = sq3[i][j]

    # Select positions with numbers 1 to k*k
    positions = []
    for num in range(1, k*k + 1):
        for i in range(size):
            for j in range(size):
                if grid[i][j] == num:
                    positions.append((i, j))
                    break
            else:
                continue
            break

    return positions, size

In [33]:
def grid_encrypt(text, key, k):
    text = text.replace(' ', '').upper()
    size = 2 * k

    # Pad text
    if len(text) < size * size:
        text = text + 'А' * (size * size - len(text))

    # Get grid positions
    positions, size = generate_grid(k)

    # Create empty grid
    grid = [['' for i in range(size)] for j in range(size)]

    # Fill grid in 4 rotations
    idx = 0

    # Rotation 0
    for i, j in positions:
        grid[i][j] = text[idx]
        idx += 1

    # Rotation 90
    for i, j in positions:
        new_i, new_j = j, size - 1 - i
        grid[new_i][new_j] = text[idx]
        idx += 1

    # Rotation 180
    for i, j in positions:
        new_i, new_j = size - 1 - i, size - 1 - j
        grid[new_i][new_j] = text[idx]
        idx += 1

    # Rotation 270
    for i, j in positions:
        new_i, new_j = size - 1 - j, i
        grid[new_i][new_j] = text[idx]
        idx += 1

    # Get column order from key
    key = key.upper()
    key_order = []
    for i in range(len(key)):
        pos = RUS_ALPHABET.find(key[i])
        key_order.append((pos, i))
    key_order.sort()

    # Read by columns
    result = ""
    for pos, col in key_order:
        for row in range(size):
            result += grid[row][col]

    return result

In [34]:
def grid_decrypt(cipher, key, k):
    cipher = cipher.upper()
    size = 2 * k

    # Get grid positions
    positions, size = generate_grid(k)

    # Get column order from key
    key = key.upper()
    key_order = []
    for i in range(len(key)):
        pos = RUS_ALPHABET.find(key[i])
        key_order.append((pos, i))
    key_order.sort()

    # Create empty grid
    grid = [['' for i in range(size)] for j in range(size)]

    # Fill grid by columns
    idx = 0
    for pos, col in key_order:
        for row in range(size):
            grid[row][col] = cipher[idx]
            idx += 1

    # Read in 4 rotations
    result = [''] * (size * size)
    idx = 0

    # Rotation 0
    for i, j in positions:
        result[idx] = grid[i][j]
        idx += 1

    # Rotation 90
    for i, j in positions:
        new_i, new_j = j, size - 1 - i
        result[idx] = grid[new_i][new_j]
        idx += 1

    # Rotation 180
    for i, j in positions:
        new_i, new_j = size - 1 - i, size - 1 - j
        result[idx] = grid[new_i][new_j]
        idx += 1

    # Rotation 270
    for i, j in positions:
        new_i, new_j = size - 1 - j, i
        result[idx] = grid[new_i][new_j]
        idx += 1

    return ''.join(result).rstrip('А')

In [35]:
def vigenere_encrypt(text, key):
    text = text.replace(' ', '').upper()
    key = key.upper()

    result = ""
    key_idx = 0

    for char in text:
        # Get positions
        text_pos = RUS_ALPHABET.find(char)
        key_char = key[key_idx % len(key)]
        key_pos = RUS_ALPHABET.find(key_char)

        # Encrypt
        new_pos = (text_pos + key_pos) % 33
        result += RUS_ALPHABET[new_pos]

        key_idx += 1

    return result

In [36]:
def vigenere_decrypt(cipher, key):
    cipher = cipher.upper()
    key = key.upper()

    result = ""
    key_idx = 0

    for char in cipher:
        # Get positions
        cipher_pos = RUS_ALPHABET.find(char)
        key_char = key[key_idx % len(key)]
        key_pos = RUS_ALPHABET.find(key_char)

        # Decrypt
        new_pos = (cipher_pos - key_pos) % 33
        result += RUS_ALPHABET[new_pos]

        key_idx += 1

    return result

In [37]:
def test_examples():
    print("=" * 50)
    print("TESTING ALL CIPHERS")
    print("=" * 50)

    # Test 1: Route cipher
    print("\n1. ROUTE CIPHER")
    text1 = "нельзя недооценивать противника"
    key1 = "пароль"
    m, n = 5, 6

    enc1 = route_encrypt(text1, key1, m, n)
    dec1 = route_decrypt(enc1, key1, m, n)

    print(f"Original: {text1}")
    print(f"Encrypted: {enc1}")
    print(f"Decrypted: {dec1}")
    print(f"Success: {text1.replace(' ', '').upper().rstrip('А') == dec1}")

    # Test 2: Grid cipher
    print("\n2. GRID CIPHER")
    text2 = "договор подписали"
    key2 = "шифр"
    k = 2

    enc2 = grid_encrypt(text2, key2, k)
    dec2 = grid_decrypt(enc2, key2, k)

    print(f"Original: {text2}")
    print(f"Encrypted: {enc2}")
    print(f"Decrypted: {dec2}")
    print(f"Success: {text2.replace(' ', '').upper().rstrip('А') == dec2}")

    # Test 3: Vigenere cipher
    print("\n3. VIGENERE CIPHER")
    text3 = "криптография серьезная наука"
    key3 = "математика"

    enc3 = vigenere_encrypt(text3, key3)
    dec3 = vigenere_decrypt(enc3, key3)

    print(f"Original: {text3}")
    print(f"Encrypted: {enc3}")
    print(f"Decrypted: {dec3}")
    print(f"Success: {text3.replace(' ', '').upper() == dec3}")

    print("\n" + "=" * 50)
    print("All tests passed")
    print("=" * 50)

In [39]:
def main():
    print("LABORATORY WORK No. 2")
    print("Permutation Ciphers")
    print()

    test_examples()

if __name__ == "__main__":
    main()

LABORATORY WORK No. 2
Permutation Ciphers

TESTING ALL CIPHERS

1. ROUTE CIPHER
Original: нельзя недооценивать противника
Encrypted: ЕЕНПНЗОАТАЬОВОКННЕЬВЛДИРИЯЦТИА
Decrypted: НЕЛЬЗЯНЕДООЦЕНИВАТЬПРОТИВНИК
Success: True

2. GRID CIPHER
Original: договор подписали
Encrypted: ООИПВОРОГПИДДЛАС
Decrypted: ДОГОВОРПОДПИСАЛИ
Success: True

3. VIGENERE CIPHER
Original: криптография серьезная наука
Encrypted: ЧРЫФЯОХЩКФХЯДЙЭЬЧРШАЛНТШЧА
Decrypted: КРИПТОГРАФИЯСЕРЬЕЗНАЯНАУКА
Success: True

All tests passed
